# Module 1: Snowpark Python DataFrames

**Snowpark ML Fundamentals - Week 1 | Lunch & Learn**

---

## Learning Objectives
- Connect to Snowflake using Snowpark Python
- Understand the Snowpark DataFrame API (lazy evaluation, pushdown)
- Perform core operations: select, filter, aggregate, join
- Compare Snowpark DataFrames vs pandas DataFrames

## Key Concept
> **Snowpark DataFrames are lazily evaluated** - they build a query plan that executes entirely
> inside Snowflake's distributed engine. No data moves to the client until you call `.collect()`,
> `.show()`, or `.to_pandas()`.

---

In [15]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1.1 Setup & Connection

In [16]:
import sys
sys.path.insert(0, '../src')

from snowpark_fundamentals.session import create_session, validate_session

# Create session from .env credentials
session = create_session(env_path='../.env')

# Validate connection
info = validate_session(session)
for key, value in info.items():
    print(f"{key:>20}: {value}")

           warehouse: "TASK_WH"
            database: "MLDS_D"
              schema: "PREDICTIONS"
                role: "MLDS_ROLE_D"
   snowflake_version: 10.5.0


## 1.2 Generate Sample Data in MLDS_D

We generate our own sample data directly in `MLDS_D.PREDICTIONS` so we don't need
access to any external shared databases.

In [17]:
from snowpark_fundamentals.data.sample_data import (
    create_sample_customers_dataset,
    create_sample_orders_dataset,
)
from snowpark_fundamentals.data.loader import explore_dataframe

# Create sample tables in MLDS_D.PREDICTIONS
customers = create_sample_customers_dataset(session, n_rows=1000)
orders = create_sample_orders_dataset(session, n_rows=10000)

print("Customers:")
print(explore_dataframe(customers))
print("\nOrders:")
print(explore_dataframe(orders))

Customers:
{'row_count': 1000, 'column_count': 6, 'columns': ['CUSTOMER_ID', 'CUSTOMER_NAME', 'AGE', 'SEGMENT', 'ACCOUNT_BALANCE', 'COUNTRY'], 'dtypes': {'CUSTOMER_ID': 'LongType()', 'CUSTOMER_NAME': 'StringType(16777216)', 'AGE': 'LongType()', 'SEGMENT': 'StringType(16777216)', 'ACCOUNT_BALANCE': 'DoubleType()', 'COUNTRY': 'StringType(16777216)'}}

Orders:
{'row_count': 10000, 'column_count': 7, 'columns': ['ORDER_ID', 'CUSTOMER_ID', 'ORDER_DATE', 'ORDER_TOTAL', 'ORDER_STATUS', 'CATEGORY', 'ITEM_COUNT'], 'dtypes': {'ORDER_ID': 'LongType()', 'CUSTOMER_ID': 'LongType()', 'ORDER_DATE': 'DateType()', 'ORDER_TOTAL': 'DoubleType()', 'ORDER_STATUS': 'StringType(16777216)', 'CATEGORY': 'StringType(16777216)', 'ITEM_COUNT': 'LongType()'}}


In [ ]:
customers.show(5)

----------------------------------------------------------------------------------------
|"CUSTOMER_ID"  |"CUSTOMER_NAME"  |"AGE"  |"SEGMENT"   |"ACCOUNT_BALANCE"  |"COUNTRY"  |
----------------------------------------------------------------------------------------
|0              |CUSTOMER_00000   |78     |BUILDING    |6162.0             |JP         |
|1              |CUSTOMER_00001   |21     |BUILDING    |8676.0             |UK         |
|2              |CUSTOMER_00002   |22     |AUTOMOBILE  |6483.0             |FR         |
|3              |CUSTOMER_00003   |79     |BUILDING    |2884.0             |DE         |
|4              |CUSTOMER_00004   |30     |HOUSEHOLD   |1664.0             |US         |
----------------------------------------------------------------------------------------



In [21]:
orders.show(5)

-----------------------------------------------------------------------------------------------------------
|"ORDER_ID"  |"CUSTOMER_ID"  |"ORDER_DATE"  |"ORDER_TOTAL"  |"ORDER_STATUS"  |"CATEGORY"   |"ITEM_COUNT"  |
-----------------------------------------------------------------------------------------------------------
|0           |956            |2025-05-14    |3176.0         |RETURNED        |CLOTHING     |9             |
|1           |62             |2025-06-25    |4371.0         |SHIPPED         |CLOTHING     |8             |
|2           |74             |2025-11-17    |3328.0         |CANCELLED       |CLOTHING     |10            |
|3           |971            |2025-10-01    |1618.0         |PROCESSING      |ELECTRONICS  |9             |
|4           |194            |2024-04-04    |1038.0         |COMPLETED       |CLOTHING     |7             |
-----------------------------------------------------------------------------------------------------------



## 1.3 Loading Data into DataFrames

There are two primary ways to create a Snowpark DataFrame:
1. **`session.table()`** - Reference an existing table
2. **`session.sql()`** - Execute a SQL query

In [22]:
from snowpark_fundamentals.data.loader import load_table, load_with_sql

# Method 1: Load from an existing table (uses current DB/schema from .env)
customers = load_table(session, 'SAMPLE_CUSTOMERS')
customers.show(5)

# Method 2: Load with SQL query
recent_orders = load_with_sql(session, """
    SELECT * FROM SAMPLE_ORDERS
    WHERE ORDER_DATE >= DATEADD(YEAR, -1, CURRENT_DATE())
    LIMIT 5000
""")
print(f"\nRecent orders: {recent_orders.count():,}")

----------------------------------------------------------------------------------------
|"CUSTOMER_ID"  |"CUSTOMER_NAME"  |"AGE"  |"SEGMENT"   |"ACCOUNT_BALANCE"  |"COUNTRY"  |
----------------------------------------------------------------------------------------
|0              |CUSTOMER_00000   |78     |BUILDING    |6162.0             |JP         |
|1              |CUSTOMER_00001   |21     |BUILDING    |8676.0             |UK         |
|2              |CUSTOMER_00002   |22     |AUTOMOBILE  |6483.0             |FR         |
|3              |CUSTOMER_00003   |79     |BUILDING    |2884.0             |DE         |
|4              |CUSTOMER_00004   |30     |HOUSEHOLD   |1664.0             |US         |
----------------------------------------------------------------------------------------


Recent orders: 4,983


## 1.4 Exploring DataFrames

Key exploration methods every participant should know:

In [23]:
# Schema inspection
print("Schema:")
for field in customers.schema.fields:
    print(f"  {field.name:30} {str(field.datatype):20}")

print(f"\nTotal rows: {customers.count():,}")

Schema:
  CUSTOMER_ID                    LongType()          
  CUSTOMER_NAME                  StringType(16777216)
  AGE                            LongType()          
  SEGMENT                        StringType(16777216)
  ACCOUNT_BALANCE                DoubleType()        
  COUNTRY                        StringType(16777216)

Total rows: 1,000


In [6]:
# Preview data with .show()
customers.show(5)

----------------------------------------------------------------------------------------
|"CUSTOMER_ID"  |"CUSTOMER_NAME"  |"AGE"  |"SEGMENT"   |"ACCOUNT_BALANCE"  |"COUNTRY"  |
----------------------------------------------------------------------------------------
|0              |CUSTOMER_00000   |78     |BUILDING    |6162.0             |JP         |
|1              |CUSTOMER_00001   |21     |BUILDING    |8676.0             |UK         |
|2              |CUSTOMER_00002   |22     |AUTOMOBILE  |6483.0             |FR         |
|3              |CUSTOMER_00003   |79     |BUILDING    |2884.0             |DE         |
|4              |CUSTOMER_00004   |30     |HOUSEHOLD   |1664.0             |US         |
----------------------------------------------------------------------------------------



In [24]:
# Descriptive statistics
customers.describe().show()

----------------------------------------------------------------------------------------------------------------------
|"SUMMARY"  |"CUSTOMER_ID"       |"CUSTOMER_NAME"  |"AGE"               |"SEGMENT"   |"ACCOUNT_BALANCE"  |"COUNTRY"  |
----------------------------------------------------------------------------------------------------------------------
|mean       |499.5               |NULL             |48.31               |NULL        |4766.745           |NULL       |
|stddev     |288.81943609632646  |NULL             |18.406104557999228  |NULL        |3019.686518273651  |NULL       |
|max        |999.0               |CUSTOMER_00999   |80.0                |MACHINERY   |9991.0             |US         |
|min        |0.0                 |CUSTOMER_00000   |18.0                |AUTOMOBILE  |-499.0             |DE         |
|count      |1000.0              |1000             |1000.0              |1000        |1000.0             |1000       |
------------------------------------------------

## 1.5 Core DataFrame Operations

### Select & Filter

In [25]:
from snowflake.snowpark import functions as F

# SELECT specific columns
selected = customers.select('CUSTOMER_ID', 'CUSTOMER_NAME', 'ACCOUNT_BALANCE', 'SEGMENT')
selected.show(5)

# FILTER rows (WHERE clause)
high_value = customers.filter(F.col('ACCOUNT_BALANCE') > 5000)
print(f"\nHigh-value customers: {high_value.count():,}")

--------------------------------------------------------------------
|"CUSTOMER_ID"  |"CUSTOMER_NAME"  |"ACCOUNT_BALANCE"  |"SEGMENT"   |
--------------------------------------------------------------------
|0              |CUSTOMER_00000   |6162.0             |BUILDING    |
|1              |CUSTOMER_00001   |8676.0             |BUILDING    |
|2              |CUSTOMER_00002   |6483.0             |AUTOMOBILE  |
|3              |CUSTOMER_00003   |2884.0             |BUILDING    |
|4              |CUSTOMER_00004   |1664.0             |HOUSEHOLD   |
--------------------------------------------------------------------


High-value customers: 484


### Aggregation (GROUP BY)

In [26]:
# Group by segment and compute statistics
segment_stats = customers.group_by('SEGMENT').agg(
    F.count('*').alias('CUSTOMER_COUNT'),
    F.avg('ACCOUNT_BALANCE').alias('AVG_BALANCE'),
    F.max('ACCOUNT_BALANCE').alias('MAX_BALANCE'),
    F.min('ACCOUNT_BALANCE').alias('MIN_BALANCE'),
).sort(F.col('AVG_BALANCE').desc())

segment_stats.show()

--------------------------------------------------------------------------------------
|"SEGMENT"   |"CUSTOMER_COUNT"  |"AVG_BALANCE"       |"MAX_BALANCE"  |"MIN_BALANCE"  |
--------------------------------------------------------------------------------------
|MACHINERY   |192               |5123.682291666667   |9983.0         |-487.0         |
|HOUSEHOLD   |208               |4878.4567307692305  |9895.0         |-484.0         |
|FURNITURE   |215               |4744.4              |9942.0         |-499.0         |
|AUTOMOBILE  |172               |4580.470930232558   |9908.0         |-474.0         |
|BUILDING    |213               |4508.882629107981   |9991.0         |-458.0         |
--------------------------------------------------------------------------------------



### Adding Computed Columns

In [27]:
# Add a derived column using CASE/WHEN
enriched = customers.with_column(
    'VALUE_TIER',
    F.when(F.col('ACCOUNT_BALANCE') > 8000, F.lit('PLATINUM'))
     .when(F.col('ACCOUNT_BALANCE') > 5000, F.lit('GOLD'))
     .when(F.col('ACCOUNT_BALANCE') > 2000, F.lit('SILVER'))
     .otherwise(F.lit('BRONZE'))
)

# Show distribution of tiers
enriched.group_by('VALUE_TIER').count().sort('COUNT', ascending=False).show()

--------------------------
|"VALUE_TIER"  |"COUNT"  |
--------------------------
|GOLD          |304      |
|SILVER        |280      |
|BRONZE        |236      |
|PLATINUM      |180      |
--------------------------



### Joins

In [28]:
# Join customers with orders
orders = load_table(session, 'SAMPLE_ORDERS')

customer_orders = customers.join(
    orders,
    customers['CUSTOMER_ID'] == orders['CUSTOMER_ID'],
    join_type='inner'
).drop(orders['CUSTOMER_ID'])  # remove duplicate join key

print(f"Joined rows: {customer_orders.count():,}")
customer_orders.select(
    'CUSTOMER_NAME', 'SEGMENT', 'ORDER_ID', 'ORDER_TOTAL', 'ORDER_STATUS'
).show(5)

Joined rows: 9,991
-----------------------------------------------------------------------------
|"CUSTOMER_NAME"  |"SEGMENT"  |"ORDER_ID"  |"ORDER_TOTAL"  |"ORDER_STATUS"  |
-----------------------------------------------------------------------------
|CUSTOMER_00956   |HOUSEHOLD  |0           |3176.0         |RETURNED        |
|CUSTOMER_00062   |BUILDING   |1           |4371.0         |SHIPPED         |
|CUSTOMER_00074   |BUILDING   |2           |3328.0         |CANCELLED       |
|CUSTOMER_00971   |FURNITURE  |3           |1618.0         |PROCESSING      |
|CUSTOMER_00194   |HOUSEHOLD  |4           |1038.0         |COMPLETED       |
-----------------------------------------------------------------------------



## 1.6 Lazy Evaluation Deep Dive

> Nothing executes until an **action** is triggered. Transformations build a query plan.

In [29]:
# This chain builds a query plan - NO execution yet!
query_plan = (
    customers
    .filter(F.col('ACCOUNT_BALANCE') > 0)
    .select('CUSTOMER_ID', 'CUSTOMER_NAME', 'ACCOUNT_BALANCE', 'SEGMENT')
    .with_column('BALANCE_CATEGORY',
        F.when(F.col('ACCOUNT_BALANCE') > 5000, F.lit('HIGH')).otherwise(F.lit('LOW'))
    )
    .group_by('SEGMENT', 'BALANCE_CATEGORY')
    .agg(F.count('*').alias('COUNT'), F.avg('ACCOUNT_BALANCE').alias('AVG_BAL'))
)

# View the generated SQL (still no execution!)
print("Generated SQL:")
print(query_plan.queries['queries'][0])

# NOW it executes:
print("\nResults:")
query_plan.show()

Generated SQL:
SELECT 
    "SEGMENT", 
    "BALANCE_CATEGORY", 
    count(1) AS "COUNT", 
    avg("ACCOUNT_BALANCE") AS "AVG_BAL"
 FROM (
 SELECT 
    "CUSTOMER_ID", 
    "CUSTOMER_NAME", 
    "ACCOUNT_BALANCE", 
    "SEGMENT", 
     CASE  WHEN ("ACCOUNT_BALANCE" > 5000) THEN 'HIGH' ELSE 'LOW' END  AS "BALANCE_CATEGORY"
 FROM SAMPLE_CUSTOMERS
 WHERE 
    ("ACCOUNT_BALANCE" > 0)
)
 GROUP BY 
    "SEGMENT", 
    "BALANCE_CATEGORY"

Results:
------------------------------------------------------------------
|"SEGMENT"   |"BALANCE_CATEGORY"  |"COUNT"  |"AVG_BAL"           |
------------------------------------------------------------------
|BUILDING    |HIGH                |88       |7688.136363636364   |
|AUTOMOBILE  |HIGH                |81       |7410.123456790124   |
|BUILDING    |LOW                 |116      |2470.7241379310344  |
|HOUSEHOLD   |LOW                 |89       |2712.808988764045   |
|MACHINERY   |HIGH                |107      |7496.803738317757   |
|FURNITURE   |HIGH   

## 1.7 Snowpark vs Pandas: When to Use Which

| Feature | Snowpark DataFrame | Pandas DataFrame |
|---------|-------------------|------------------|
| **Execution** | Server-side (Snowflake) | Client-side (local) |
| **Scale** | Terabytes+ | Memory-limited |
| **Evaluation** | Lazy | Eager |
| **Use case** | Production ML, large data | EDA, small data, visualization |
| **Interop** | `.to_pandas()` for conversion | Load into Snowpark via `session.create_dataframe()` |

### Rule of Thumb
- **> 1 GB of data**: Use Snowpark DataFrames
- **Small data + visualization**: Convert to pandas with `.to_pandas()`
- **ML training**: Always use Snowpark ML (runs in Snowflake)

In [13]:
# Convert to pandas for visualization (small dataset only!)
segment_pd = segment_stats.to_pandas()
print(segment_pd)
print(f"\nType: {type(segment_pd)}")

      SEGMENT  CUSTOMER_COUNT  AVG_BALANCE  MAX_BALANCE  MIN_BALANCE
0   MACHINERY             192  5123.682292       9983.0       -487.0
1   HOUSEHOLD             208  4878.456731       9895.0       -484.0
2   FURNITURE             215  4744.400000       9942.0       -499.0
3  AUTOMOBILE             172  4580.470930       9908.0       -474.0
4    BUILDING             213  4508.882629       9991.0       -458.0

Type: <class 'pandas.core.frame.DataFrame'>


## Key Takeaways

1. **Snowpark DataFrames are lazy** - they push all computation to Snowflake
2. **No data moves to the client** unless you call `.collect()`, `.show()`, or `.to_pandas()`
3. The API mirrors **pandas/PySpark** - familiar for Python data engineers
4. Use `functions as F` for column operations (F.col, F.when, F.avg, etc.)
5. **All SQL operations** (filter, join, group_by) are available as DataFrame methods
6. All output tables are written to **MLDS_D.PREDICTIONS** (configured via `.env`)

---

**Next: [02 - Data Preprocessing](02_data_preprocessing.ipynb)**

In [14]:
# Clean up
session.close()